### 获取CAMELS数据

在开始正式实现 LSTM-CAMELS 之前，我们需要先获取 CAMELS 数据。为了方便大家使用 CAMELS 数据，我们已经将数据下载到平台服务器，并打包了读取 CAMELS 数据的代码，且内置在当前的 Python 环境中，所以可以直接通过简单的调用来读取 CAMELS 数据。

在开始前需要设置一个配置文件，指明下载的 CAMELS 数据集所在目录。如果未配置，导入 `hydrodataset` 可能会提示未找到配置文件。请先按说明完成设置。



### NetCDF 格式简介

我们会经常遇见 `.nc` 文件，也就是 NetCDF 格式文件。

从数学上来说，NetCDF 存储的数据就是**多个多自变量的单值函数**。一个函数用公式来说就是 `f(x, y, z, …) = value`。

- 函数的自变量 x, y, z 等在 NetCDF 中叫做**维 (dimension)** 或**坐标轴 (axis)**
- 函数值 value 在 NetCDF 中叫做**变量 (Variables)**

一个 NetCDF 文件的结构包括以下对象：
- **变量 (Variables)**：变量对应着真实的物理数据
- **维 (dimension)**：一个维对应着函数中的某个自变量，或者说函数图象中的一个坐标轴，典型地是三维（经纬度+时间）
- **属性 (Attribute)**：属性是对变量值和维的具体物理含义的注释

NetCDF 文件中的数据以数组形式存储。例如：
- 某个位置处随时间变化的温度以一维数组的形式存储
- 某个区域内在指定时间的温度以二维数组的形式存储
- 三维 (3D) 数据（如某个区域内随时间变化的温度）或四维 (4D) 数据（如某个区域内随时间和高度变化的温度）以一系列二维数组的形式存储


### 使用 xarray 读取 NetCDF 文件

读取 `.nc` 文件有很多种方法，这里演示 Python 中最常用的使用 `xarray` 读取的方式。

我们通过 `xarray` 工具包中的 `.open_dataset()` 函数来读取 `.nc` 文件。

具体使用的数据是公开数据集 CAMELS 的流域径流时间序列数据，它原本格式是 txt 的，为了方便使用，我们已经将它处理到 nc 格式并放在平台服务器了，现在我们直接读取它。


### 配置 hydrodataset 数据目录

首次使用需要配置 CAMELS 数据集的文件路径。

#### 方法1：使用服务器（如 jupyterhub）

在平台 jupyterlab 首页打开终端，然后输入以下命令：

```bash
# 进入配置文件所在文件夹
cd ~/.hydrodataset
# 使用 vim 打开配置文件
vim settings.txt
```

打开后，按 `i` 键，将 vim 编辑器调整至 INSERT 模式，然后输入 `/ftproot`（ftproot 是服务器上放置公共数据的默认文件夹）。

然后按 `:` （英文输入法下的冒号）进入命令模式，输入 `wq` 并按回车键，就能写入（即保存）并退出了。

配置完成后，需要重新打开当前文件的 kernel，重新执行下面的导入语句。


In [ ]:
# 导入 hydrodataset 配置包
import hydrodataset

# 检查数据目录是否正确
print("数据根目录:", hydrodataset.ROOT_DIR)
# 显示'/ftproot'即正确

### 读取 CAMELS NetCDF 数据示例

现在数据已经配置好了，我们就能读取一个 nc 文件试试了。


In [ ]:
import xarray as xr

dataset_dir = hydrodataset.ROOT_DIR
camels_streamflow = xr.open_dataset(
    dataset_dir.joinpath("camels", "camels_us", "camels_streamflow.nc")
)
camels_streamflow


可以看见 NetCDF 文件的维度 (Dimensions)、坐标 (Coordinates)、属性 (Attributes) 与变量 (Data variables)。

查看有哪些流域：


In [ ]:
# 查看流域列表
camels_streamflow.basin


成功后，接下来我们就可以试试读取CAMELS数据集了

In [ ]:
# 配置与导入（如未配置会报错，已配置可直接使用）
import hydrodataset
from hydrodataset.camels import Camels
import os

# 如未配置，将出现配置文件未找到的提示；已配置则可忽略此段提示



In [ ]:
# 创建 CAMELS-US 数据对象（示例）
camels_us_path = os.path.join("camels", "camels_us")
us_region = "US"
camels_us = Camels(camels_us_path, region=us_region)
camels_us.camels_sites.head()


### 使用缓存的高效数据格式

- 径流与气象数据采用 NetCDF（nc）格式，建议用 `xarray` 懒加载，避免一次性加载到内存。
- 属性数据采用 feather 格式，可用 `pandas` 读取。

下方示例展示数据目录、缓存目录，以及如何读取流量与气象数据集。


In [ ]:
import pandas as pd
import xarray as xr

# 数据与缓存目录
data_dir = camels_us.data_source_dir
cache_dir = hydrodataset.CACHE_DIR
print("data_dir:", data_dir)
print("cache_dir:", cache_dir)

# 读取数据集
streamflow_ds = xr.open_dataset(data_dir.joinpath("camels_streamflow.nc"))
forcing_ds = xr.open_dataset(data_dir.joinpath("camels_daymet_forcing.nc"))
attrs = pd.read_feather(data_dir.joinpath("camels_attributes_v2.0.feather"))

streamflow_ds, forcing_ds, attrs.head()


In [ ]:
# 示例：选择一个流域与时间段进行可视化（如需）
_ = streamflow_ds.sel(basin="01013500", time=slice("2000-06-01", "2001-05-31")).to_pandas().plot()
